# PKS-MPNN Dataset Exploration

This notebook explores the PKS module dataset:
- Structure counts by architecture type
- Sequence length distributions
- Domain type frequencies
- Overview of the training data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from src.data.annotation_parser import AnnotationParser, CORE_DOMAINS

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load annotations
annotation_csv = Path('../fragments_for_prediction_COREONLY.csv')
parser = AnnotationParser(annotation_csv)
print(f'Loaded {len(parser)} module annotations')

## Domain Composition Distribution

In [ ]:
# Get composition counts
compositions = parser.get_composition_counts()
top_compositions = sorted(compositions.items(), key=lambda x: x[1], reverse=True)[:20]

fig, ax = plt.subplots(figsize=(14, 6))
names, counts = zip(*top_compositions)
ax.barh(range(len(names)), counts)
ax.set_yticks(range(len(names)))
ax.set_yticklabels([n[:50] + '...' if len(n) > 50 else n for n in names], fontsize=8)
ax.set_xlabel('Count')
ax.set_title('Top 20 Domain Compositions')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'Total unique compositions: {len(compositions)}')

## Domain Type Frequencies

In [ ]:
# Count domain types
domain_counts = parser.get_domain_type_counts()

# Separate core domains and linkers
core_counts = {k: v for k, v in domain_counts.items() if k in CORE_DOMAINS}
linker_counts = {k: v for k, v in domain_counts.items() if k not in CORE_DOMAINS}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Core domains
ax = axes[0]
sorted_core = sorted(core_counts.items(), key=lambda x: x[1], reverse=True)
names, counts = zip(*sorted_core) if sorted_core else ([], [])
ax.bar(names, counts, color='steelblue')
ax.set_xlabel('Domain Type')
ax.set_ylabel('Count')
ax.set_title('Core Domain Frequencies')
ax.tick_params(axis='x', rotation=45)

# Linkers
ax = axes[1]
sorted_linker = sorted(linker_counts.items(), key=lambda x: x[1], reverse=True)[:15]
names, counts = zip(*sorted_linker) if sorted_linker else ([], [])
ax.bar(names, counts, color='coral')
ax.set_xlabel('Linker Type')
ax.set_ylabel('Count')
ax.set_title('Top 15 Linker Frequencies')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Sequence Length Distribution

In [ ]:
# Get sequence lengths
lengths = [ann.length for ann in parser]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(lengths, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Sequence Length (residues)')
ax.set_ylabel('Count')
ax.set_title('Sequence Length Distribution')
ax.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean: {np.mean(lengths):.0f}')
ax.axvline(np.median(lengths), color='green', linestyle='--', label=f'Median: {np.median(lengths):.0f}')
ax.legend()

# Box plot by composition category
ax = axes[1]
length_by_type = {}
for ann in parser:
    n_domains = len(ann.core_domains)
    key = f'{n_domains} domains'
    if key not in length_by_type:
        length_by_type[key] = []
    length_by_type[key].append(ann.length)

data = [length_by_type[k] for k in sorted(length_by_type.keys())]
labels = sorted(length_by_type.keys())
ax.boxplot(data, labels=labels)
ax.set_xlabel('Number of Core Domains')
ax.set_ylabel('Sequence Length')
ax.set_title('Length by Number of Domains')

plt.tight_layout()
plt.show()

print(f'Length statistics:')
print(f'  Min: {min(lengths)}')
print(f'  Max: {max(lengths)}')
print(f'  Mean: {np.mean(lengths):.1f}')
print(f'  Median: {np.median(lengths):.1f}')
print(f'  Std: {np.std(lengths):.1f}')

## Summary Statistics

In [ ]:
# Compute summary
total_residues = sum(lengths)
unique_seqs = len(parser.get_unique_sequences())

print('Dataset Summary')
print('=' * 40)
print(f'Total modules:           {len(parser):,}')
print(f'Unique sequences:        {unique_seqs:,}')
print(f'Total residues:          {total_residues:,}')
print(f'Unique compositions:     {len(compositions):,}')
print(f'Core domain types:       {len(core_counts)}')
print(f'Linker types:            {len(linker_counts)}')